<a id="top"></a>

# Demo: a tool call is just tokens

```yaml
title: "Demo: a tool call is just tokens"
keywords: tool call, bind_tools, tool_calls, tool definition, schema, reasoning is tokens, protocol, teacher demo
estimated duration: 10
```

> **Lesson:** L07. Demo 1 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L07/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running. A real model **varies**: the exact wording may differ run to run, but the model should reach for the tool most of the time. Clear outputs before committing.
>
> **The client is LangChain's `ChatAnthropic`** — the same framework client L03 introduced and every lesson since has used. `model.bind_tools([calculator])` lets a plain typed Python function *be* the tool: LangChain infers the definition (name, description, per-argument JSON schema) from the function and hands it to the model. The model answers with an `AIMessage` whose `.tool_calls` list is the structured request. You'll stop the moment it arrives and inspect it — you do **not** run the calculator here.
>
> The point to land: a tool call is **tokens the model emitted**, not an action it took.

## Contents

- [1. Setup](#1-setup)
- [2. No tools registered — the L01–L06 world](#2-no-tools-registered--the-l01l06-world)
- [3. Register the tool — and stop at the block](#3-register-the-tool--and-stop-at-the-block)
- [4. What just happened (and didn't)](#4-what-just-happened-and-didnt)
- [5. Takeaways](#5-takeaways)

## 1. Setup

The LangChain `ChatAnthropic` client, the single `calculator` tool, and that tool **bound** to the model with `bind_tools`. Anchor model: **Claude Sonnet 4.6**.

In [ ]:
import json
from typing import Annotated

from langchain_anthropic import ChatAnthropic
from langchain_core.utils.function_calling import convert_to_openai_tool

from fluffy_potato_curriculum.common.config import require_anthropic_key

SONNET = "claude-sonnet-4-6"


# The ONE tool that carries all four demos: a deterministic calculator.
# L07 is deliberately single-tool, single-round-trip (multi-tool is L08, the
# agent loop is L10). Resist adding a second tool.
#
# There is no hand-written "tool definition" here: bind_tools INFERS it from this
# function. The name comes from `calculator`, the description from the docstring,
# and the argument schema from the type hint -- the Annotated string becomes the
# argument's JSON-Schema description. A plain typed function IS the schema.
def calculator(
    expression: Annotated[str, "The arithmetic expression to evaluate, e.g. '18374 * 92431'."],
) -> str:
    """Evaluate a basic arithmetic expression (the four operators and parentheses) and
    return the exact numeric result. Use this whenever the user asks for the result of
    a calculation.

    Restricted to digits and the operators + - * / ( ) . and whitespace so a
    hallucinated expression cannot run arbitrary code. Raises ValueError on
    anything else -- that rejection is the whole point of Demo 4."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the character whitelist above blocks names,
    # attributes, and calls. Never eval untrusted input without such a guard.
    return str(eval(expression))


# Constructing the client raises a clear error if ANTHROPIC_API_KEY is missing
# (the key is read through common.config, not a raw os.environ lookup).
model = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=400)

# bind_tools returns a NEW model handle that carries the tool definition on every
# call. The bare `model` (no tools) still exists -- we use both below to contrast.
model_with_tools = model.bind_tools([calculator])


# A prompt the model is unreliable at without a calculator, so it has a real
# reason to reach for the tool.
PROMPT = "What is 18,374 multiplied by 92,431?"
print("setup ready (live model:", SONNET, ")")

[↑ Back to top](#top)

## 2. No tools registered — the L01–L06 world

First, the exact prompt with **no tools**. The model answers directly, often with a confident, wrong number. This is text-in, text-out — everything L01–L06 did.

In [ ]:
# The bare model, no tools -- exactly the L01-L06 world: text in, text out.
no_tools = model.invoke(PROMPT)
print(no_tools.content)

[↑ Back to top](#top)

## 3. Register the tool — and stop at the block

Now the tool-bound model (`model_with_tools`) and the **same** prompt. First, print the definition LangChain inferred and hands to the model — this dict, **not** the Python function, is all the model ever sees. Then invoke and stop the moment the response arrives: you do **not** run the calculator. Read `.content` (any text the model wrote) and `.tool_calls` (its structured requests) side by side.

In [ ]:
# The definition bind_tools inferred from the function -- name, description, and the
# per-argument JSON schema. THIS dict rides along in the prompt on every call; the
# Python function stays on our side and the model never sees it.
print(json.dumps(convert_to_openai_tool(calculator)["function"], indent=2))

In [ ]:
resp = model_with_tools.invoke(PROMPT)

print("text content:", repr(resp.content))
print("tool_calls in this one response:")
for call in resp.tool_calls:
    print(f"  name={call['name']!r}  id={call['id']!r}  args={call['args']!r}")

[↑ Back to top](#top)

## 4. What just happened (and didn't)

Read the tool call aloud: a **name** (`calculator`), an **args** dict (`{'expression': ...}`), and a unique **id**. Nothing was computed — the model did not multiply anything. It emitted a *structured request* that says "I would like the calculator run with this expression." The number inside `args` came from the model's tokens, not from arithmetic.

In [ ]:
call = resp.tool_calls[0]
print("the model PROPOSED this call (it did not run it):")
print("  name      :", call["name"])
print("  arguments :", call["args"])
print("  call id   :", call["id"])
print("\nThree actors, only one has acted: the MODEL proposed; the APPLICATION (us)")
print("holds the tool call and has done nothing; the TOOL (calculator) has not run.")

[↑ Back to top](#top)

## 5. Takeaways

- A tool call is **just more structured tokens** — the same shape-contract idea as L06's `<thinking>`/`<answer>` tags, except now your application is expected to *act* on the shape. It arrives as an entry in `AIMessage.tool_calls`: a **name**, an **args** dict, and an **id**.
- **Three actors**, only one has moved: model proposed, application holds the tool call, tool has not run. The rest of the lesson fills in the remaining steps.
- **`bind_tools` inferred the definition** from the plain typed function — a function's name, docstring, and type hints *are* the schema the model sees.
- The decision to emit a tool call *instead of* answering directly is itself a **reasoning step** — the same one you built in L06. A vague tool is one the model skips ([L08](L0702_lecture.md) answers *why*).
- Next: [Demo 2](L0704_lecture.ipynb) completes the loop this demo stopped halfway through.

[↑ Back to top](#top)